[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/fast_track/09_apis_and_sql.ipynb)

# 📓 Notebook 9 (fast track) — APIs & SQL: Getting Real Data In

> **Module:** Real-World I/O · **Estimated time:** 75–95 min · **Difficulty:** Beginner → Intermediate

So far the data has come from variables and CSVs you were handed. Real work means **fetching** it yourself — over HTTP from web **APIs**, and out of relational databases with **SQL**. These are the two front doors to almost every dataset you'll ever use. This notebook keeps the essentials of both.


> 🏎️ **You're on the fast track.** This is a trimmed, **combined** notebook that condenses two canonical chapters into one: [`03_real_world_io/12_apis_and_http.ipynb`](../03_real_world_io/12_apis_and_http.ipynb) (APIs & HTTP) and [`03_real_world_io/13_sql_fundamentals.ipynb`](../03_real_world_io/13_sql_fundamentals.ipynb) (SQL). From each, the Stretch exercises (A–D) and the 🎁 Bonus mini-project have been removed to keep the path lean — the core teaching, every Practice exercise, and Stretch C/D are kept. Open the canonical versions once you want the deeper material.

---


# 🌐 Part 1 — APIs & HTTP  *(canonical NB 12)*


## 1. The shape of every HTTP request

```
                   ┌─── METHOD: GET | POST | PUT | DELETE
                   │   ┌── URL
                   │   │
                   │   │            ┌── HEADERS  (auth, content-type, …)
                   │   │            │
                   ▼   ▼            ▼
GET https://api.example.com/v1/weather   {"Authorization": "Bearer ..."}
                                          │
                          QUERY PARAMS    │  BODY  (POST/PUT only)
                          ?lat=52.5       │  {"name": "Alice"}
                          &lon=13.4       ▼
```

The four pieces above (method, URL, headers, body) describe **every** HTTP request you'll ever make — to a weather API, an LLM, a CRM, anything. The `requests` library wraps all of this in one function call per method: `requests.get(...)`, `requests.post(...)`, etc.

## 2. Setup

In [ ]:
import requests
import pandas as pd
from typing import Optional
import json
import time

print(f"requests version: {requests.__version__}")
print(f"pandas   version: {pd.__version__}")

# Two APIs we'll use throughout — both free, no key needed
WEATHER_API = "https://api.open-meteo.com/v1/forecast"
PLACEHOLDER  = "https://jsonplaceholder.typicode.com"

# A short timeout so a hanging network can't lock up our notebook
HTTP_TIMEOUT = 8   # seconds


## 3. Your first GET request

Let's ask the Open-Meteo API for the *current temperature in Berlin*.

In [ ]:
# 1. Build the parameters: latitude, longitude, what we want to know
params = {
    "latitude":  52.52,    # Berlin
    "longitude": 13.41,
    "current":   "temperature_2m",
}

# 2. Make the request
try:
    response = requests.get(WEATHER_API, params=params, timeout=HTTP_TIMEOUT)
except requests.exceptions.RequestException as e:
    print(f"⚠ Network unavailable ({type(e).__name__}). Using a recorded response.")
    payload = {
        "current": {"time": "2024-01-01T12:00", "temperature_2m": 4.3},
        "current_units": {"temperature_2m": "°C"},
    }
else:
    print(f"Status code: {response.status_code} ({response.reason})")
    print(f"URL called : {response.url}")
    payload = response.json()

# 3. Pull the field we care about
temp = payload["current"]["temperature_2m"]
unit = payload["current_units"]["temperature_2m"]
print(f"\nCurrent temperature in Berlin: {temp} {unit}")


**What just happened.**

1. `requests.get(url, params=...)` builds the URL `https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&current=temperature_2m` and sends a GET request.
2. The server returns a JSON document. `response.json()` parses it into a Python dict.
3. We drill into the dict with `[...]` (NB 3 patterns) to pull the value we want.

> 💡 **The `try / except` wrapping a real network call is non-negotiable.** Networks fail. Timeouts happen. APIs go down. Always have a fallback path — at minimum, fail loudly instead of hanging the kernel.

## 4. HTTP status codes — what the server is telling you

| Code range | Meaning              | Examples |
|------------|----------------------|----------|
| **2xx**    | Success              | `200 OK`, `201 Created`, `204 No Content` |
| **3xx**    | Redirection          | `301 Moved Permanently`, `304 Not Modified` |
| **4xx**    | *You* did something wrong | `400 Bad Request`, `401 Unauthorized`, `404 Not Found`, `429 Too Many Requests` |
| **5xx**    | *Server* did something wrong | `500 Internal Server Error`, `502 Bad Gateway`, `503 Service Unavailable` |

The single most important habit: **check the status code before using the response.** `requests` gives you two ways:

In [ ]:
# A) Check explicitly. Wrap the live call so the cell still runs offline.
try:
    r = requests.get(f"{PLACEHOLDER}/posts/1", timeout=HTTP_TIMEOUT)
    if r.status_code == 200:
        print(f"OK: got {len(r.text)} bytes")
    else:
        print(f"Error: {r.status_code}")
except requests.exceptions.RequestException:
    print("Offline — skipping live check")

# B) Use raise_for_status() — raises HTTPError for 4xx / 5xx
try:
    r = requests.get(f"{PLACEHOLDER}/this-does-not-exist", timeout=HTTP_TIMEOUT)
    r.raise_for_status()
except requests.HTTPError as e:
    print(f"\nHTTPError caught: {e}")
except requests.exceptions.RequestException as e:
    print(f"\nOther network error: {type(e).__name__}: {e}")


> 🎯 **Rule of thumb.** Use `r.raise_for_status()` inside a `try / except requests.HTTPError` block. It keeps your business logic clean — the moment any 4xx / 5xx arrives, control jumps to the error handler.

## 5. Headers, authentication, and `User-Agent`

Most production APIs require some form of authentication via a request **header**. The two patterns you'll see 95% of the time:

```python
# Bearer token (OpenAI, Anthropic, GitHub, most modern APIs)
headers = {"Authorization": f"Bearer {API_KEY}"}

# API key in a custom header (older APIs)
headers = {"X-API-Key": API_KEY}

requests.get(url, headers=headers, timeout=8)
```

Free public APIs like Open-Meteo don't require auth, but they often *do* appreciate a user-agent so they know who's calling. Let's set one.

In [ ]:
# Setting custom headers — same pattern you'll use for bearer-token auth
import os

headers = {
    # Identify yourself politely
    "User-Agent": "python-for-ai-course/1.0 (learning HTTP requests)",
    # When you eventually need a real API:
    # "Authorization": f"Bearer {os.getenv('OPENAI_API_KEY', 'sk-...')}"
}

try:
    r = requests.get(f"{PLACEHOLDER}/users/1", headers=headers, timeout=HTTP_TIMEOUT)
    r.raise_for_status()
    user = r.json()
    print(f"User #{user['id']}: {user['name']}")
    print(f"  email: {user['email']}")
    print(f"  city : {user['address']['city']}")
except requests.exceptions.RequestException as e:
    print(f"⚠ Network unavailable. Recorded value: 'Leanne Graham'")


> ⚠️ **Never put API keys in your notebook or git history.** Read them from environment variables (`os.getenv('OPENAI_API_KEY')`) or a `.env` file with `python-dotenv`. The comment line above shows the pattern.

## 6. POST — sending data to a server

`POST` is for *creating* something. The data goes in the request **body**, usually as JSON.

In [ ]:
# Create a new post on the fake-API server
new_post = {
    "title":  "What I learned about HTTP today",
    "body":   "Status codes, headers, retries, pagination. All of it.",
    "userId": 1,
}

try:
    r = requests.post(f"{PLACEHOLDER}/posts", json=new_post, timeout=HTTP_TIMEOUT)
    r.raise_for_status()
    created = r.json()
    print(f"Created post #{created.get('id')} (status {r.status_code} {r.reason})")
    print(f"  title: {created.get('title')}")
except requests.exceptions.RequestException as e:
    print(f"⚠ Network unavailable. (POST would have returned id=101)")


> 💡 `json=new_post` tells `requests` to JSON-encode your dict and set the right `Content-Type` header. The alternative `data=...` parameter sends form-encoded data instead.

## 7. Retry with exponential backoff

Networks are *unreliable*. A polite client retries with increasing delays — usually `0.5s → 1s → 2s → 4s → ...`. This is exactly the pattern from NB2, dressed up for HTTP.

In [ ]:
def fetch_with_retry(url: str,
                     params: Optional[dict] = None,
                     max_attempts: int = 4,
                     base_delay: float = 0.5) -> Optional[dict]:
    """GET a URL with exponential backoff. Returns the JSON payload or None."""
    for attempt in range(1, max_attempts + 1):
        try:
            r = requests.get(url, params=params, timeout=HTTP_TIMEOUT)
            # 429 = rate-limited, 5xx = server problem — both worth retrying
            if r.status_code == 429 or 500 <= r.status_code < 600:
                raise requests.HTTPError(f"server said {r.status_code}", response=r)
            r.raise_for_status()
            return r.json()
        except (requests.exceptions.RequestException, requests.HTTPError) as e:
            if attempt == max_attempts:
                print(f"  attempt {attempt}: giving up ({e})")
                return None
            wait = base_delay * 2 ** (attempt - 1)
            print(f"  attempt {attempt}: {e}  — waiting {wait:.2f}s")
            time.sleep(wait)
    return None


# Try a few endpoints — the second one is intentionally broken
for url in [f"{PLACEHOLDER}/users/1", f"{PLACEHOLDER}/this-route-does-not-exist"]:
    print(f"\nGET {url}")
    data = fetch_with_retry(url, max_attempts=2)
    print(f"  result: {type(data).__name__}{' (ok)' if data else ' (failed)'}")


> 🎯 **Which errors do you retry?**
> - Retry: **network timeouts**, **429 Too Many Requests**, **5xx** server errors.
> - **Don't** retry: **400 Bad Request**, **401 Unauthorized**, **404 Not Found** — these will fail again on the next try. Fix the request instead.

## 8. Pagination — when one request isn't enough

Most APIs return at most ~100 items per request. To get more, you **paginate** — making one request per page until you've seen everything.

Two common pagination schemes:

| Scheme | How it works | API examples |
|---|---|---|
| **Page number** | `?page=1&per_page=50`, then `?page=2`, … | GitHub, many REST APIs |
| **Cursor**       | response has a `next_cursor`; pass it in the next call | Twitter/X, Slack, modern APIs |

We'll demo the page-number pattern.

In [ ]:
def fetch_all_pages(base_url: str, per_page: int = 10, max_pages: int = 20) -> list:
    """Fetch every page from a page-number-style endpoint."""
    all_items = []
    for page in range(1, max_pages + 1):
        try:
            r = requests.get(base_url,
                             params={"_page": page, "_limit": per_page},
                             timeout=HTTP_TIMEOUT)
            r.raise_for_status()
            items = r.json()
        except requests.exceptions.RequestException:
            print(f"  page {page}: network error, stopping")
            break
        if not items:                # empty page → we're done
            print(f"  page {page}: empty, stopping")
            break
        all_items.extend(items)
        print(f"  page {page}: fetched {len(items)} items (total {len(all_items)})")
    return all_items


# JSONPlaceholder has 100 posts; we'll grab them 25 at a time
posts = fetch_all_pages(f"{PLACEHOLDER}/posts", per_page=25)
print(f"\n→ Total posts fetched: {len(posts)}")


> 💡 **Always cap your loop.** `max_pages=20` is a safety net. Without it, a buggy API that always returns the same page would loop forever.

## 9. Putting it together — a real ETL pipeline

Let's build a tiny **E**xtract-**T**ransform-**L**oad job:

1. **Extract**: fetch a 7-day weather forecast for several cities.
2. **Transform**: flatten the response into one tidy row per (city, day).
3. **Load**: save to a CSV that any later notebook can read.

In [ ]:
CITIES = [
    ("Berlin",    52.52, 13.41),
    ("London",    51.51, -0.13),
    ("New York",  40.71, -74.01),
    ("Tokyo",     35.68, 139.65),
]

def fetch_forecast(city: str, lat: float, lon: float) -> Optional[pd.DataFrame]:
    """Get a 7-day daily forecast for one city, return as a DataFrame."""
    params = {
        "latitude": lat, "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "auto",
        "forecast_days": 7,
    }
    data = fetch_with_retry(WEATHER_API, params=params, max_attempts=3)
    if data is None or "daily" not in data:
        return None
    df = pd.DataFrame(data["daily"])
    df["city"] = city
    return df

# Run the pipeline
frames = []
for city, lat, lon in CITIES:
    print(f"→ Fetching forecast for {city} ...")
    f = fetch_forecast(city, lat, lon)
    if f is not None:
        frames.append(f)

if frames:
    forecast = pd.concat(frames, ignore_index=True)
    forecast = forecast[["city", "time", "temperature_2m_max",
                          "temperature_2m_min", "precipitation_sum"]]
    print(f"\n✅ Combined DataFrame: {forecast.shape}")
    print(forecast.head(10))
else:
    print("⚠ No data fetched (offline?) — try again with internet.")


In [ ]:
# Save the result so other notebooks can use it
if frames:
    out_path = "forecast.csv"
    forecast.to_csv(out_path, index=False)
    print(f"Wrote {out_path}  ({len(forecast)} rows)")


**Read that pipeline carefully.** It is short, but it is *real*:

- One function per responsibility (`fetch_forecast` extracts + does an initial transform).
- The retry / fallback logic from earlier is reused without modification.
- The output is a clean CSV that integrates with everything else you've learned (NB 6).

This is the canonical shape of every data-pull script you will ever write. The only thing that changes between projects is the URL and the schema.

## 10. Common pitfalls

| Pitfall | Symptom | Fix |
|---|---|---|
| No timeout on the request | notebook hangs forever | Always pass `timeout=...` |
| API key in the code | leaks to git history | Use `os.getenv(...)` |
| Hammering an API in a tight loop | get rate-limited (`429`) | Add `time.sleep(...)` between calls |
| Parsing JSON without checking status | `JSONDecodeError` on error pages | Call `r.raise_for_status()` first |
| One bad row kills the whole batch | exception crashes the loop | Wrap each call in `try / except` |
| Retrying a 401 / 404 | wastes time, won't change | Retry only for `429` and `5xx` |

## 🧪 Practice exercises

### Exercise 1 — ⭐ A simple `safe_fetch`

Write `safe_fetch(url)` that:

1. Sends a GET with `timeout=8`.
2. Returns the parsed JSON on `200`.
3. Returns `None` (no crash) on any error.
4. Prints a one-line warning so a human knows what happened.

Test it on `f"{PLACEHOLDER}/users/2"` (works) and `f"{PLACEHOLDER}/no-such-thing"` (404).

In [ ]:
# Your code here  👇
def safe_fetch(url):
    pass


<details>
<summary>💡 <b>Solution</b></summary>

```python
def safe_fetch(url: str):
    try:
        r = requests.get(url, timeout=HTTP_TIMEOUT)
        r.raise_for_status()
        return r.json()
    except requests.HTTPError as e:
        print(f"⚠ HTTP {e.response.status_code} for {url}")
    except requests.exceptions.RequestException as e:
        print(f"⚠ Network error: {type(e).__name__}")
    return None

print(safe_fetch(f"{PLACEHOLDER}/users/2"))     # works
print(safe_fetch(f"{PLACEHOLDER}/no-such-thing"))  # 404 → None, no crash
```

**Reasoning.** A defensive wrapper at the boundary of your code (where it meets the outside world) keeps the rest of your script simple. Errors are visible (the warning print) but non-fatal.
</details>

### Exercise 2 — ⭐⭐ Build a tiny user report

Use the JSONPlaceholder API (`{PLACEHOLDER}/users`) to:

1. Fetch all users (no pagination needed — the endpoint returns 10).
2. Build a DataFrame with columns `id`, `name`, `email`, `city`, `company`.
3. Print the unique cities they live in.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
data = safe_fetch(f"{PLACEHOLDER}/users") or []

rows = [{
    "id":      u["id"],
    "name":    u["name"],
    "email":   u["email"],
    "city":    u["address"]["city"],
    "company": u["company"]["name"],
} for u in data]

df_users = pd.DataFrame(rows)
print(df_users)
print(f"\nUnique cities: {df_users['city'].unique().tolist()}")
```

**Pattern used.** Flatten nested JSON into a flat row dict with a list comprehension. We'll see this exact idiom whenever an API returns deeply-nested objects — it converts JSON-land into table-land in one expression.
</details>

### Exercise 3 — ⭐⭐ Rate-limited polite client

Wrap `safe_fetch` in `polite_fetch_all(urls, delay=0.2)` that calls each URL with a small delay between calls — so you never hammer the server. Then fetch posts 1–5 from JSONPlaceholder.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def polite_fetch_all(urls, delay=0.2):
    results = []
    for i, url in enumerate(urls):
        if i > 0:
            time.sleep(delay)
        results.append(safe_fetch(url))
    return results

urls = [f"{PLACEHOLDER}/posts/{i}" for i in range(1, 6)]
posts = polite_fetch_all(urls, delay=0.1)
for p in posts:
    if p:
        print(f"#{p['id']}: {p['title'][:50]}…")
```

**Why this matters.** Every API has a rate limit — some explicit (Twitter, Slack), some implicit (Open-Meteo will silently throttle you above a few requests per second). A small `time.sleep` between calls is the difference between *useful client* and *blocked client*.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

The function below is supposed to return the title of a JSONPlaceholder post, but it sometimes crashes mysteriously. Find the bug(s).

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
def get_title(post_id):
    r = requests.get(f"{PLACEHOLDER}/posts/{post_id}")
    return r.json()["title"]

# This call works...
print(get_title(1))
# ...but this one fails with a confusing error if the server is slow or the id is bad.
# print(get_title(99999))


<details>
<summary>💡 <b>Solution</b></summary>

Three problems:

1. **No timeout.** A slow server would hang the notebook indefinitely.
2. **No status-code check.** `requests.get(.../posts/99999)` returns `404` with an empty `{}` body — `r.json()["title"]` then raises `KeyError`, not a clear HTTP error.
3. **No error handling.** A network blip propagates as an obscure traceback.

```python
def get_title(post_id):
    try:
        r = requests.get(f"{PLACEHOLDER}/posts/{post_id}", timeout=HTTP_TIMEOUT)
        r.raise_for_status()
        return r.json().get("title")     # .get() returns None if missing
    except requests.exceptions.RequestException as e:
        print(f"⚠ Failed to fetch post {post_id}: {e}")
        return None
```

**Lesson.** Every external call has three failure modes — *slow*, *unavailable*, *unexpected response*. A 4-line function should defend against all three.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise C — ⭐⭐⭐ Defensive JSON parser

API responses lie. Write a function `safe_get(d, *keys, default=None)` that walks a nested dict by a sequence of keys and returns `default` if any key is missing or any value along the way isn't a dict.

```python
data = {"user": {"profile": {"name": "Ada"}}}
safe_get(data, "user", "profile", "name")     # → 'Ada'
safe_get(data, "user", "settings", "theme")    # → None
safe_get(data, "user", "profile", "name", "first")  # → None  (name is a string, not a dict)
```

In [ ]:
# Your code here  👇
def safe_get(d, *keys, default=None):
    ...

data = {"user": {"profile": {"name": "Ada"}}}
print(safe_get(data, "user", "profile", "name"))
print(safe_get(data, "user", "settings", "theme"))


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def safe_get(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d
```

**Reasoning.** Three details worth absorbing. (1) The `isinstance(d, dict)` guard catches the case where you descend into a non-dict value (a string, a list, `None`) — without it you'd raise `TypeError`. (2) Reassigning `d = d[k]` walks the structure one key at a time — concise and intuitive. (3) Returning a single `default` sentinel beats raising in API-parsing code: callers can use `or` (`name = safe_get(d, ...) or "anonymous"`) or compare against the sentinel as they prefer. The standard library's `dict.get` only handles one level; this is the multi-level version you'll write again and again.
</details>

### Stretch exercise D — ⭐⭐⭐ Paginate through an API

Many APIs return data in pages with a `next_url` field, e.g.:

```python
page_1 = {"results": [1, 2, 3],   "next_url": "https://api.example.com/items?page=2"}
page_2 = {"results": [4, 5, 6],   "next_url": "https://api.example.com/items?page=3"}
page_3 = {"results": [7, 8],      "next_url": None}
```

Write `collect_all_pages(fetch, start_url)` where `fetch(url)` returns a page dict. It should return the **flat** list of all results across pages. To keep this offline, use the stub `fetch` in the skeleton.

In [ ]:
# Your code here  👇
PAGES = {
    "https://api/p1": {"results": [1, 2, 3],  "next_url": "https://api/p2"},
    "https://api/p2": {"results": [4, 5, 6],  "next_url": "https://api/p3"},
    "https://api/p3": {"results": [7, 8],     "next_url": None},
}
def fetch(url): return PAGES[url]

def collect_all_pages(fetch, start_url):
    ...

# print(collect_all_pages(fetch, "https://api/p1"))


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def collect_all_pages(fetch, start_url):
    items = []
    url = start_url
    while url:
        page = fetch(url)
        items.extend(page["results"])
        url = page.get("next_url")
    return items

print(collect_all_pages(fetch, "https://api/p1"))  # [1,2,3,4,5,6,7,8]
```

**Reasoning.** Pagination is one of those patterns you'll write more times than you'd think. Two design choices to internalise. (1) Take `fetch` as a parameter rather than calling `requests.get` directly — this makes the function trivially testable with a fake fetcher (which is exactly what we did here). (2) Loop on `url` rather than counting pages — the API tells you when to stop by returning `next_url = None`. Don't invent your own stop condition (`page < 100`) unless you also log a warning when it triggers; runaway loops in production rack up real API bills.
</details>

## 🧠 Key takeaways

1. Every HTTP request is **method + URL + headers + (optional body)**. The `requests` library wraps it in one function call per method.
2. **Always pass `timeout=...`** — without it, your code can hang forever.
3. **Always check the status code** with `r.raise_for_status()` before using the response.
4. Retry **429** and **5xx**; never retry **4xx** (other than 429) — fix the request instead.
5. Use **environment variables** for API keys, never inline them.
6. **`json=...`** for POST bodies, **`params=...`** for query strings, **`headers=...`** for auth.
7. The shape of every real-world data-fetch script: **extract → transform → load** (ETL).
8. Defensive wrappers at the **boundary** of your code keep the **inside** simple.

## ✅ Self-assessment

- [ ] Make a GET request with query parameters and headers
- [ ] Read and act on HTTP status codes
- [ ] Use `raise_for_status()` inside a `try / except`
- [ ] Send a POST request with a JSON body
- [ ] Write a retry loop with exponential backoff for transient errors
- [ ] Paginate through a multi-page result set
- [ ] Flatten a nested JSON response into a flat DataFrame

# 🗄️ Part 2 — SQL Fundamentals  *(canonical NB 13)*


## 1. SQL in one slide

SQL describes data manipulation as a sequence of **clauses** that operate on tables. The most common shape:

```
SELECT   columns_or_aggregations         ← what you want
FROM     table                            ← where to look
WHERE    row_filter                       ← which rows
GROUP BY column(s)                        ← bucket rows together
HAVING   aggregate_filter                 ← which buckets
ORDER BY column(s)                        ← sort the result
LIMIT    n                                ← top-n only
```

You won't use every clause every time. A simple "show me the first ten rows" is just `SELECT * FROM table LIMIT 10`.

The mental model: **SQL describes *what* you want, the database figures out *how*.** That declarative style is the source of both its power and its quirks.

### 🔬 What actually happens — SQL runs clauses in a *different* order than you write them

The slide above lists the clauses in **writing order** (`SELECT … FROM … WHERE … GROUP BY … HAVING … ORDER BY … LIMIT`). But the database **does not execute them top-to-bottom.** It follows a fixed *logical execution order* — and once you internalise it, half of SQL's "quirks" become obvious.

```
   WRITTEN ORDER                 LOGICAL EXECUTION ORDER (what the DB does)
   ─────────────                 ─────────────────────────────────────────
   SELECT    ──┐                 1. FROM / JOIN   ── assemble the rows
   FROM        │                 2. WHERE         ── filter raw rows
   WHERE       │   the DB        3. GROUP BY      ── bucket rows together
   GROUP BY    │   re-orders     4. HAVING        ── filter the buckets
   HAVING      │   to ───────▶   5. SELECT        ── compute columns & aliases
   ORDER BY    │                 6. ORDER BY      ── sort the result
   LIMIT     ──┘                 7. LIMIT         ── keep the top-n
```

> 🎯 **The one consequence to remember:** `SELECT` runs at step **5**, but `WHERE` runs at step **2** — *before* it. So a column **alias you define in `SELECT` does not exist yet when `WHERE` runs.** That single fact explains the most common beginner error in SQL. (SQLite is the lenient exception — it *does* let `WHERE` use the alias — but Postgres and MySQL reject it, so don't rely on it; the proof cell below shows both.)


In [ ]:
# PROOF (stdlib sqlite3, in-memory, no network): the alias / WHERE interaction.
# IMPORTANT: SQLite is LENIENT — unlike standard SQL it DOES let WHERE see a SELECT alias.
import sqlite3

con = sqlite3.connect(":memory:")
con.executescript("""
    CREATE TABLE t (name TEXT, price REAL, qty INTEGER);
    INSERT INTO t VALUES ('a', 10.0, 3), ('b', 5.0, 1), ('c', 20.0, 2);
""")

# ⚠️ STANDARD SQL (Postgres/MySQL) rejects this: WHERE runs at step 2, before SELECT
# (step 5) defines the `total` alias — you'd get "no such column: total".
# But SQLite ACCEPTS it as a non-standard convenience, so it runs and returns rows:
rows = con.execute("SELECT name, price*qty AS total FROM t WHERE total > 15").fetchall()
print("WHERE total    -> (SQLite allows it):", rows)   # [('a', 30.0), ('c', 40.0)]

# ✅ The portable form — repeat the expression so it works in EVERY dialect (it's
# evaluated from the raw columns at step 2, no alias needed):
rows = con.execute("SELECT name, price*qty AS total FROM t WHERE price*qty > 15").fetchall()
print("WHERE price*qty ->", rows)             # [('a', 30.0), ('c', 40.0)]

# ✅ ORDER BY runs at step 6, AFTER SELECT — so the alias is visible there in ALL dialects:
rows = con.execute("SELECT name, price*qty AS total FROM t ORDER BY total DESC").fetchall()
print("ORDER BY total ->", rows)              # [('c', 40.0), ('a', 30.0), ('b', 5.0)]
con.close()


In [ ]:
# PROOF #2: WHERE vs HAVING — the same split. WHERE filters RAW ROWS (step 2),
# HAVING filters GROUPS (step 4), so only HAVING can see an aggregate.
con = sqlite3.connect(":memory:")
con.executescript("""
    CREATE TABLE sales (region TEXT, amount REAL);
    INSERT INTO sales VALUES
        ('north', 100), ('north', 50), ('south', 30), ('south', 20), ('east', 200);
""")

# ❌ WHERE cannot see SUM(amount) — the GROUP BY (step 3) hasn't happened yet at step 2:
try:
    con.execute("SELECT region, SUM(amount) FROM sales WHERE SUM(amount) > 80 GROUP BY region")
except sqlite3.OperationalError as e:
    print("WHERE SUM(...) -> ERROR:", e)      # misuse of aggregate

# ✅ HAVING runs at step 4, after grouping, so it CAN filter on the aggregate:
rows = con.execute(
    "SELECT region, SUM(amount) AS total FROM sales GROUP BY region HAVING SUM(amount) > 80"
).fetchall()
print("HAVING SUM(...) ->", rows)             # [('east', 200.0), ('north', 150.0)]
con.close()


> 🧠 **Carry this one mental model.** Read every query in execution order, not writing order: *first assemble rows (`FROM`/`JOIN`), then filter rows (`WHERE`), then bucket (`GROUP BY`), then filter buckets (`HAVING`), then compute columns (`SELECT`), then sort (`ORDER BY`), then trim (`LIMIT`).* It instantly answers the two questions that trip up everyone — **"why can't `WHERE` use my `SELECT` alias?"** (alias born later) and **"`WHERE` or `HAVING`?"** (raw rows → `WHERE`; aggregated groups → `HAVING`). You'll see the `HAVING` version of this exact trap again in the Debug-me exercise below.

> ⚠️ Two dialect nuances: (1) most databases (PostgreSQL, MySQL, SQLite) let `ORDER BY` reference a `SELECT` alias, since it runs *after* `SELECT`. (2) In **standard SQL** `WHERE` can *never* use a `SELECT` alias, because it is logically evaluated before `SELECT`. **SQLite is the lenient exception** and accepts it anyway (see the proof cell) — but the portable habit is to repeat the expression in `WHERE`, which works everywhere.


## 2. Setup — load a CSV into SQLite

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

# Build a fresh synthetic dataset inline (so the notebook is self-contained)
import numpy as np
RNG = np.random.default_rng(42)

channels = ["Email", "Chat", "Phone", "Web Form", "Social"]
months   = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
profile = {
    "Email":    (0.55, 0.018, 4_200,  3_500, 4.10),
    "Chat":     (0.72, 0.012, 6_500,    900, 4.35),
    "Phone":    (0.18, 0.008, 2_100, 14_000, 3.85),
    "Web Form": (0.65, 0.015, 3_300,  6_000, 4.00),
    "Social":   (0.35, 0.030, 1_400,  2_400, 3.95),
}
rows = []
for ch in channels:
    a0, g, vol, base_lat, base_sat = profile[ch]
    for i, m in enumerate(months):
        a = max(0.05, min(0.95, a0 + i * g + RNG.normal(0, 0.015)))
        v = int(vol * (1 + 0.01 * i + RNG.normal(0, 0.03)))
        lat = base_lat * (1 - 0.012 * i) * (1 + RNG.normal(0, 0.05))
        sat = max(1.0, min(5.0, base_sat + 0.4 * (a - a0)
              - 0.000_004 * (lat - base_lat) + RNG.normal(0, 0.05)))
        cost = max(0.15, a * 0.30 + (1 - a) * 5.50 + RNG.normal(0, 0.10))
        rows.append({"channel": ch, "month": m, "month_num": i+1,
                     "tickets_total": v, "tickets_auto": int(v*a),
                     "automation_rate": round(a, 3),
                     "latency_ms": int(lat), "satisfaction": round(sat, 2),
                     "cost_per_ticket": round(cost, 2)})
support_ops = pd.DataFrame(rows)

# Create an in-memory SQLite database (no file on disk — instant)
conn = sqlite3.connect(":memory:")

# Push the DataFrame into a table
support_ops.to_sql("support_ops", conn, index=False, if_exists="replace")

print(f"✅ Loaded {len(support_ops)} rows into table 'support_ops'")
print(f"   Columns: {list(support_ops.columns)}")


**What just happened.**

- `sqlite3.connect(":memory:")` opens a database that exists only in RAM — perfect for teaching.
- `df.to_sql("name", conn)` is the pandas → SQL bridge. Reverses with `pd.read_sql(...)`, shown next.

> 💡 **In real life** you'd use `sqlite3.connect("ops.db")` to persist the database to a file, or `sqlalchemy.create_engine("postgresql://...")` for a real production database. The rest of this notebook is identical regardless.

## 3. Your first SELECT

The `pd.read_sql(query, conn)` function runs a SQL query and gives you the result as a DataFrame. Let's use it to peek at the table.

In [ ]:
# The simplest possible query — "give me the first 5 rows"
q = "SELECT * FROM support_ops LIMIT 5"
pd.read_sql(q, conn)


In [ ]:
# Just the columns we care about
q = '''
SELECT channel, month, automation_rate, satisfaction
FROM support_ops
LIMIT 5
'''
pd.read_sql(q, conn)


**Conventions worth picking up immediately:**

- **Uppercase keywords** (`SELECT`, `FROM`, `WHERE`) and **lowercase identifiers** (`channel`, `support_ops`). The DB doesn't care, but humans read SQL faster when it follows this convention.
- **One clause per line** for anything longer than a one-liner. Vertical SQL is much easier to debug.

## 4. WHERE — filtering rows

`WHERE` is SQL's row filter — same job as a pandas boolean mask.

In [ ]:
# All months where the bot was working very well (high automation, high satisfaction)
q = '''
SELECT channel, month, automation_rate, satisfaction
FROM   support_ops
WHERE  automation_rate >= 0.80
   AND satisfaction    >= 4.30
ORDER BY automation_rate DESC
'''
pd.read_sql(q, conn)


In [ ]:
# String comparisons — note the single quotes around the literal
q = '''
SELECT channel, month, latency_ms
FROM   support_ops
WHERE  channel = 'Phone'
   AND latency_ms > 10000
ORDER BY latency_ms DESC
LIMIT 5
'''
pd.read_sql(q, conn)


**SQL operators that show up daily:**

| Operator | Meaning | Example |
|---|---|---|
| `=`, `<>`, `<`, `>`, `<=`, `>=` | comparisons | `WHERE channel = 'Chat'` |
| `AND`, `OR`, `NOT` | combine conditions | `WHERE a > 5 AND b < 10` |
| `IN (...)` | match any of a list | `WHERE channel IN ('Email','Chat')` |
| `BETWEEN x AND y` | range | `WHERE month_num BETWEEN 4 AND 6` |
| `LIKE 'pat%'` | string pattern (`%` = wildcard) | `WHERE channel LIKE 'E%'` |
| `IS NULL` / `IS NOT NULL` | missing values | `WHERE satisfaction IS NOT NULL` |

> ⚠️ **Use `IS NULL`, not `= NULL`.** SQL's three-valued logic (`TRUE` / `FALSE` / `UNKNOWN`) treats `column = NULL` as always *unknown*, so it never matches anything.

## 5. Aggregations — the part that earns its keep

The five core aggregation functions — `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` — combined with `GROUP BY` produce most of the analytical output a business consumes.

In [ ]:
# Total tickets and average automation rate, per channel
q = '''
SELECT channel,
       COUNT(*)               AS n_months,
       SUM(tickets_total)     AS total_tickets,
       AVG(automation_rate)   AS mean_auto_rate,
       AVG(satisfaction)      AS mean_satisfaction
FROM     support_ops
GROUP BY channel
ORDER BY total_tickets DESC
'''
pd.read_sql(q, conn).round(3)


**Reading the query above:**

- `GROUP BY channel` buckets the 60 rows into 5 groups (one per channel).
- Each `AGG(...)` is computed *per bucket*.
- `AS column_name` renames the output — much clearer than the default `AVG(automation_rate)`.

> 🎯 **The rule of `GROUP BY`.** Every column in the `SELECT` list must either be in the `GROUP BY` clause **or** be wrapped in an aggregation function. Mixing the two without grouping is a beginner bug the database will (eventually) refuse to run.

### `HAVING` — filtering on aggregates

`WHERE` filters *rows*. `HAVING` filters *groups*. You usually want one or the other, sometimes both.

In [ ]:
# Channels with average automation rate > 60%
q = '''
SELECT channel,
       AVG(automation_rate) AS mean_auto
FROM     support_ops
GROUP BY channel
HAVING   AVG(automation_rate) > 0.60
ORDER BY mean_auto DESC
'''
pd.read_sql(q, conn).round(3)


## 6. Grouping by multiple columns and time

`GROUP BY` can take several columns — useful when you want a cross-tab.

In [ ]:
# Quarterly automation rate per channel
q = '''
SELECT channel,
       CASE
         WHEN month_num <= 3 THEN 'Q1'
         WHEN month_num <= 6 THEN 'Q2'
         WHEN month_num <= 9 THEN 'Q3'
         ELSE 'Q4'
       END                          AS quarter,
       AVG(automation_rate)         AS mean_auto
FROM     support_ops
GROUP BY channel, quarter
ORDER BY channel, quarter
'''
pd.read_sql(q, conn).round(3)


> 💡 **`CASE WHEN ... THEN ... ELSE ... END`** is SQL's if-elif-else. It runs once per row and returns a value. We just used it to turn a continuous month number into a categorical quarter — feature engineering in pure SQL.

## 7. JOINs — bringing two tables together

Real databases have many tables. Imagine we have a separate table of **channel metadata** — the team manager, the channel's launch year, and so on. Let's create one and join it back.

In [ ]:
# A small "dimension" table about each channel
channel_meta = pd.DataFrame([
    {"channel": "Email",    "team_lead": "Anna",  "launched_year": 2018, "is_voice": 0},
    {"channel": "Chat",     "team_lead": "Bilal", "launched_year": 2020, "is_voice": 0},
    {"channel": "Phone",    "team_lead": "Carla", "launched_year": 2010, "is_voice": 1},
    {"channel": "Web Form", "team_lead": "Diego", "launched_year": 2019, "is_voice": 0},
    {"channel": "Social",   "team_lead": "Elena", "launched_year": 2023, "is_voice": 0},
])
channel_meta.to_sql("channel_meta", conn, index=False, if_exists="replace")

# Now join: each row of support_ops gets the team_lead and launched_year attached
q = '''
SELECT s.channel,
       s.month,
       s.automation_rate,
       m.team_lead,
       m.launched_year
FROM   support_ops AS s
JOIN   channel_meta AS m ON s.channel = m.channel
WHERE  s.month_num = 12         -- December only
ORDER BY s.automation_rate DESC
'''
pd.read_sql(q, conn).round(3)


**The four kinds of JOIN, in plain English:**

| Type | Keeps … |
|---|---|
| `INNER JOIN` (default) | rows that match in **both** tables |
| `LEFT JOIN`            | every row from the left, with `NULL` where the right has no match |
| `RIGHT JOIN`           | the mirror image (rare in modern SQL — flip your tables) |
| `FULL OUTER JOIN`      | every row from either side; matches where possible |

> 🎯 **99% of the time you want `INNER JOIN` or `LEFT JOIN`.** Reach for `LEFT JOIN` whenever the right table might be missing rows for some keys and you don't want to drop those rows.

In [ ]:
# A LEFT JOIN — useful when the right table might be missing entries
extra = pd.DataFrame([{"channel": "Email", "monthly_budget_usd": 12000}])
extra.to_sql("budgets", conn, index=False, if_exists="replace")

q = '''
SELECT s.channel,
       AVG(s.cost_per_ticket) AS mean_cost,
       b.monthly_budget_usd
FROM     support_ops AS s
LEFT JOIN budgets AS b ON s.channel = b.channel
GROUP BY s.channel
'''
pd.read_sql(q, conn).round(3)


See how channels without a budget show `None` instead of being dropped. That's the `LEFT JOIN` paying its way — you preserve all channels even when the budget table only knows about one.

## 8. CTEs — making complex queries readable

A **CTE** (Common Table Expression) lets you give a name to an intermediate query and reuse it. It is the SQL equivalent of breaking a long Python function into helper functions.

In [ ]:
# Without a CTE — works, but hard to read
q_inline = '''
SELECT channel, mean_auto, mean_sat
FROM (
  SELECT channel,
         AVG(automation_rate) AS mean_auto,
         AVG(satisfaction)    AS mean_sat
  FROM   support_ops
  GROUP BY channel
)
WHERE mean_auto > 0.5 AND mean_sat > 4.0
'''
print("Inline subquery result:")
print(pd.read_sql(q_inline, conn).round(3))

# Same query with a CTE — much easier to read and to extend
q_cte = '''
WITH channel_summary AS (
    SELECT channel,
           AVG(automation_rate) AS mean_auto,
           AVG(satisfaction)    AS mean_sat
    FROM   support_ops
    GROUP BY channel
)
SELECT *
FROM   channel_summary
WHERE  mean_auto > 0.5 AND mean_sat > 4.0
ORDER BY mean_auto DESC
'''
print("\nCTE result:")
print(pd.read_sql(q_cte, conn).round(3))


> 💡 **When in doubt, CTE.** A 5-line CTE that you read top-to-bottom always beats a 5-level-deep nested subquery. Modern SQL style guides recommend CTEs for anything beyond a single `SELECT`.

## 9. SQL vs pandas — when to use which

You can do almost any analysis in either tool. So when do you reach for which?

| Use **SQL** when … | Use **pandas** when … |
|---|---|
| The data lives in a database — *don't move it before you have to*. | The data is already a DataFrame. |
| Joining 2–10 large tables on keys. | Reshaping a single dataset (pivot, melt, multi-index). |
| Filtering / aggregating "most" of a large table before bringing it into Python. | Doing per-row, per-column transformations that are awkward in SQL. |
| Sharing the query with non-Python colleagues. | Plugging into matplotlib, scikit-learn, etc. |
| The result needs to be a stable view that anyone can re-run. | Quick exploration, prototyping. |

> 🎯 **A common pro pattern:** *use SQL for the heavy lift, pandas for the last mile.* Pull a small filtered/aggregated table out of the database with SQL, then do the plotting / modelling in pandas. You get the best of both.

In [ ]:
# Same analysis, two ways — pick the one that reads better
# A) Pure SQL
print("--- SQL ---")
print(pd.read_sql('''
    SELECT channel,
           AVG(automation_rate) AS mean_auto
    FROM   support_ops
    GROUP BY channel
    ORDER BY mean_auto DESC
''', conn).round(3))

# B) Pure pandas
print("\n--- pandas ---")
print(support_ops.groupby("channel")["automation_rate"]
                 .mean().sort_values(ascending=False)
                 .round(3).to_frame())


For this tiny query they look equally readable. For a 4-table join with three CTEs and a window function — SQL wins on clarity. For a chained `.assign(...).pivot(...).rolling(...).plot()` — pandas wins.

## 10. Tiny tour of window functions

Window functions compute an aggregate **per row** using a "window" of nearby rows — running totals, ranks, lag/lead. They are SQL's secret weapon.

In [ ]:
# Rank each channel's months from best to worst by automation rate
q = '''
SELECT channel,
       month,
       automation_rate,
       RANK() OVER (PARTITION BY channel ORDER BY automation_rate DESC) AS auto_rank
FROM support_ops
ORDER BY channel, auto_rank
'''
pd.read_sql(q, conn).head(15)


The output has three months per channel where `auto_rank = 1` would be the best month for that channel, etc. That's a *ranking inside each group* — a pattern that costs you a `groupby + transform` dance in pandas but one line of SQL.

Window-function shapes worth knowing:

- `RANK() OVER (...)` — competition ranking (ties share a rank).
- `ROW_NUMBER() OVER (...)` — unique 1, 2, 3, … even for ties.
- `SUM(x) OVER (PARTITION BY g ORDER BY t)` — running total within each group.
- `LAG(x, 1) OVER (ORDER BY t)` — value from the previous row.

## 11. Cleaning up

In [ ]:
conn.close()
print("Connection closed.")


## 🧪 Practice exercises

### Exercise 1 — ⭐ Recreate the connection and run a query

Reopen the connection, recreate the `support_ops` table from the DataFrame, then write a query that returns the **mean cost-per-ticket per channel**, sorted ascending.

In [ ]:
# Your code here  👇
conn = sqlite3.connect(":memory:")
support_ops.to_sql("support_ops", conn, index=False, if_exists="replace")


<details>
<summary>💡 <b>Solution</b></summary>

```python
q = '''
SELECT channel,
       AVG(cost_per_ticket) AS mean_cost
FROM     support_ops
GROUP BY channel
ORDER BY mean_cost ASC
'''
pd.read_sql(q, conn).round(2)
```

Chat should come out cheapest, Phone most expensive. The pattern is **`SELECT … FROM … GROUP BY … ORDER BY`** — the four clauses you'll use in every other analytical query you write.
</details>

### Exercise 2 — ⭐⭐ A two-condition filter

Write a query that returns every (channel, month) where **automation_rate > 0.75 AND latency_ms < 2000**. Sort by automation_rate descending.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
q = '''
SELECT channel, month, automation_rate, latency_ms
FROM   support_ops
WHERE  automation_rate > 0.75
   AND latency_ms      < 2000
ORDER BY automation_rate DESC
'''
pd.read_sql(q, conn).round(3)
```

These are the "fast + accurate" months — your *high-leverage* combinations of conditions.
</details>

### Exercise 3 — ⭐⭐ Best month per channel (window function)

Use a window function to find each channel's **single best month** (highest satisfaction). The result should have exactly 5 rows.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
q = '''
WITH ranked AS (
    SELECT channel, month, satisfaction,
           ROW_NUMBER() OVER (PARTITION BY channel
                              ORDER BY satisfaction DESC) AS rn
    FROM support_ops
)
SELECT channel, month, satisfaction
FROM ranked
WHERE rn = 1
ORDER BY satisfaction DESC
'''
pd.read_sql(q, conn).round(2)
```

**Pattern.** Rank rows inside each group, then keep only `rn = 1`. This "top-N per group" pattern is a window-function classic — and the reason every senior analyst learns them.
</details>

### Exercise 4 — ⭐⭐ Join with the metadata table

Recreate `channel_meta` (as we did in §7), then write a query that returns the **mean automation rate** per channel together with its **team lead** and **launched year**, sorted by automation rate descending.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
channel_meta = pd.DataFrame([
    {"channel": "Email",    "team_lead": "Anna",  "launched_year": 2018},
    {"channel": "Chat",     "team_lead": "Bilal", "launched_year": 2020},
    {"channel": "Phone",    "team_lead": "Carla", "launched_year": 2010},
    {"channel": "Web Form", "team_lead": "Diego", "launched_year": 2019},
    {"channel": "Social",   "team_lead": "Elena", "launched_year": 2023},
])
channel_meta.to_sql("channel_meta", conn, index=False, if_exists="replace")

q = '''
SELECT s.channel,
       AVG(s.automation_rate) AS mean_auto,
       m.team_lead,
       m.launched_year
FROM     support_ops AS s
JOIN     channel_meta AS m ON s.channel = m.channel
GROUP BY s.channel, m.team_lead, m.launched_year
ORDER BY mean_auto DESC
'''
pd.read_sql(q, conn).round(3)
```

**Pattern used.** Join → group → aggregate → sort. This is the shape of every "X by Y, enriched with Z" report your stakeholders will ever ask for.
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞

The query below is supposed to return channels with **more than 50K total tickets**, but it raises an error. Find the bug.

```sql
SELECT channel
FROM   support_ops
WHERE  SUM(tickets_total) > 50000
GROUP BY channel
```

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
# Your fixed query  👇


<details>
<summary>💡 <b>Solution</b></summary>

`WHERE` filters individual rows **before** aggregation, so it cannot reference an aggregate like `SUM(...)`. Filtering on aggregates is the job of `HAVING`, which runs **after** the group is formed.

```python
q = '''
SELECT channel,
       SUM(tickets_total) AS total
FROM   support_ops
GROUP BY channel
HAVING SUM(tickets_total) > 50000
ORDER BY total DESC
'''
pd.read_sql(q, conn)
```

**Remember the order of operations in SQL:** `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT`. Knowing this order makes 90% of confusing errors instantly obvious.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise C — ⭐⭐⭐ Top-N per group

Using an in-memory SQLite DB with a `sales` table (columns: `region`, `product`, `revenue`), find the **top-2 products by revenue within each region**.

Hint: window functions. SQLite supports `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`.

In [ ]:
# Your code here  👇
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE sales (region TEXT, product TEXT, revenue REAL);
INSERT INTO sales VALUES
  ('EU', 'A', 120), ('EU', 'B', 90),  ('EU', 'C', 150), ('EU', 'D',  60),
  ('US', 'A', 200), ('US', 'B', 180), ('US', 'C', 100), ('US', 'D', 220);
""")

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE sales (region TEXT, product TEXT, revenue REAL);
INSERT INTO sales VALUES
  ('EU', 'A', 120), ('EU', 'B', 90),  ('EU', 'C', 150), ('EU', 'D',  60),
  ('US', 'A', 200), ('US', 'B', 180), ('US', 'C', 100), ('US', 'D', 220);
""")

sql = """
WITH ranked AS (
    SELECT region, product, revenue,
           ROW_NUMBER() OVER (PARTITION BY region ORDER BY revenue DESC) AS rk
    FROM   sales
)
SELECT region, product, revenue FROM ranked WHERE rk <= 2 ORDER BY region, rk;
"""
for row in con.execute(sql):
    print(row)
```

**Reasoning.** Top-N-per-group is the canonical use case for window functions. `ROW_NUMBER() OVER (PARTITION BY region ORDER BY revenue DESC)` numbers rows within each region starting at 1 from the highest revenue. We wrap the query in a CTE (`WITH ranked AS …`) so we can then filter by `rk <= 2` — you can't refer to a window-function alias in the same SELECT's `WHERE`, which is why the CTE is needed. If you ever need to allow ties, swap `ROW_NUMBER()` for `RANK()` (ties share a rank, gaps follow) or `DENSE_RANK()` (no gaps).
</details>

### Stretch exercise D — ⭐⭐⭐ Add a column and back-fill

You have a `users` table with `id` and `email`. Add a new column `email_domain` and back-fill it with everything after the `@` in each existing email. Then add a constraint that future inserts without an `email_domain` are rejected.

Hint: in SQLite you use `ALTER TABLE … ADD COLUMN`, then a single `UPDATE … SET … = SUBSTR(...)`.

In [ ]:
# Your code here  👇
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE users (id INTEGER PRIMARY KEY, email TEXT);
INSERT INTO users VALUES
  (1, 'ada@example.com'),
  (2, 'linus@kernel.org'),
  (3, 'grace@navy.mil');
""")

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE users (id INTEGER PRIMARY KEY, email TEXT);
INSERT INTO users VALUES
  (1, 'ada@example.com'),
  (2, 'linus@kernel.org'),
  (3, 'grace@navy.mil');
""")

# Step 1: add the new column (nullable by default in SQLite)
con.execute("ALTER TABLE users ADD COLUMN email_domain TEXT")

# Step 2: back-fill
con.execute("""
    UPDATE users
    SET    email_domain = SUBSTR(email, INSTR(email, '@') + 1)
""")

for row in con.execute("SELECT * FROM users"):
    print(row)

# Step 3: enforce non-null going forward (SQLite can't add a NOT NULL
# to an existing column directly — recreate the table or use a trigger).
con.execute("""
    CREATE TRIGGER users_domain_not_null
    BEFORE INSERT ON users
    FOR EACH ROW WHEN NEW.email_domain IS NULL
    BEGIN SELECT RAISE(ABORT, 'email_domain must be set'); END;
""")
```

**Reasoning.** This is the bread-and-butter of *schema migration*. Three notes. (1) `ADD COLUMN` is one of the few `ALTER TABLE` operations SQLite supports — most other RDBMS (Postgres, MySQL) are more flexible. (2) `SUBSTR(email, INSTR(email, '@') + 1)` is the SQLite way of writing `email.split('@')[1]`. (3) SQLite can't add `NOT NULL` to an existing column without rewriting the table; a `BEFORE INSERT` trigger gives you the same end-user effect with less surgery. In Postgres you'd use `ALTER TABLE … ALTER COLUMN … SET NOT NULL` after the back-fill.
</details>

## 🧠 Key takeaways

1. **SQL is declarative**: you describe *what* you want, the database figures out *how*.
2. The six core clauses — `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `ORDER BY`, `LIMIT` — cover most analytics work.
3. **`HAVING` filters aggregates; `WHERE` filters rows.** Use them in the right place.
4. **Every column in `SELECT` must be in `GROUP BY` or wrapped in an aggregation.**
5. **`INNER JOIN`** for "must match"; **`LEFT JOIN`** for "may be missing".
6. **CTEs (`WITH ... AS`)** make complex queries readable — use them freely.
7. **Window functions** let you compute aggregates per row — the secret weapon for top-N-per-group and running totals.
8. **SQL vs pandas:** SQL for joins and heavy aggregation; pandas for reshape, plot, model. They're complements, not rivals.

## ✅ Self-assessment

- [ ] Load a DataFrame into a SQLite table and read it back as a DataFrame
- [ ] Write a `SELECT ... WHERE ... ORDER BY ... LIMIT` query
- [ ] Aggregate with `GROUP BY` and the five basic aggregation functions
- [ ] Filter aggregates with `HAVING`
- [ ] Inner-join and left-join two tables
- [ ] Use a CTE (`WITH ...`) to structure a multi-step query
- [ ] Explain when SQL beats pandas and vice versa

## 🚀 Next step

Continue with **Notebook 10 (fast track) — AI-Assisted Workflows**, where you start calling LLMs like ordinary Python functions and wiring them into pipelines.